<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Lab: Custom Training Loops in Keras**


Estimated time needed: **30** minutes


In this lab, you will learn to implement a basic custom training loop in Keras. 


## Objectives

By the end of this lab, you will: 

- Set up the environment 

- Define the neural network model 

- Define the Loss Function and Optimizer 

- Implement the custom training loop 

- Enhance the custom training loop by adding an accuracy metric to monitor model performance 

- Implement a custom callback to log additional metrics and information during training


----


## Step-by-Step Instructions:


### Exercise 1: Basic custom training loop: 

#### 1. Set Up the Environment:

- Import necessary libraries. 

- Load and preprocess the MNIST dataset. 


In [ ]:
# !pip install tensorflow numpy

In [1]:
import os
import warnings
import tensorflow as tf 
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.callbacks import Callback
import numpy as np

# Suppress all Python warnings
warnings.filterwarnings('ignore')

# Set TensorFlow log level to suppress warnings and info messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data() 
x_train, x_test = x_train / 255.0, x_test / 255.0 
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)


2026-04-14 23:20:14.677970: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-14 23:20:14.778127: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-14 23:20:14.778617: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-14 23:20:14.940260: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-14 23:20:16.840137: W tensorflow/compiler/tf

#### 2. Define the model: 

Create a simple neural network model with a Flatten layer followed by two Dense layers. 


In [2]:
# Step 2: Define the Model

model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='relu'),
    Dense(10)
])


#### 3. Define Loss Function and Optimizer: 

- Use Sparse Categorical Crossentropy for the loss function. 
- Use the Adam optimizer. 


In [3]:
# Step 3: Define Loss Function and Optimizer

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) 
optimizer = tf.keras.optimizers.Adam()


#### 4. Implement the Custom Training Loop: 

- Iterate over the dataset for a specified number of epochs. 
- Compute the loss and apply gradients to update the model's weights. 


In [4]:
# Step 4: Implement the Custom Training Loop

epochs = 2
# train_dataset = train_dataset.repeat(epochs)
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)
for epoch in range(epochs):
    print(f'Start of epoch {epoch + 1}')

    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            logits = model(x_batch_train, training=True)  # Forward pass
            loss_value = loss_fn(y_batch_train, logits)  # Compute loss

        # Compute gradients and update weights
        grads = tape.gradient(loss_value, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))

        # Logging the loss every 200 steps
        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()}')


Start of epoch 1
Epoch 1 Step 0: Loss = 2.3577218055725098
Epoch 1 Step 200: Loss = 0.4053861200809479
Epoch 1 Step 400: Loss = 0.18480730056762695
Epoch 1 Step 600: Loss = 0.15610134601593018
Epoch 1 Step 800: Loss = 0.14808818697929382
Epoch 1 Step 1000: Loss = 0.4142986536026001
Epoch 1 Step 1200: Loss = 0.1820199340581894
Epoch 1 Step 1400: Loss = 0.28537338972091675
Epoch 1 Step 1600: Loss = 0.25651901960372925
Epoch 1 Step 1800: Loss = 0.14788642525672913


2026-04-14 23:46:02.070491: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Start of epoch 2
Epoch 2 Step 0: Loss = 0.08173525333404541
Epoch 2 Step 200: Loss = 0.1997140496969223
Epoch 2 Step 400: Loss = 0.11112159490585327
Epoch 2 Step 600: Loss = 0.0304560624063015
Epoch 2 Step 800: Loss = 0.11809844523668289
Epoch 2 Step 1000: Loss = 0.23568633198738098
Epoch 2 Step 1200: Loss = 0.09526168555021286
Epoch 2 Step 1400: Loss = 0.19261905550956726
Epoch 2 Step 1600: Loss = 0.20424166321754456
Epoch 2 Step 1800: Loss = 0.06842992454767227


2026-04-14 23:46:51.204088: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### Exercise 2: Adding Accuracy Metric:

Enhance the custom training loop by adding an accuracy metric to monitor model performance. 

#### 1. Set Up the Environment: 

Follow the setup from Exercise 1. 


In [5]:
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Flatten 

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize the pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0 

# Create a batched dataset for training
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)


#### 2. Define the Model: 
Use the same model as in Exercise 1. 


In [6]:
# Step 2: Define the Model

model = Sequential([ 
    Flatten(input_shape=(28, 28)),  # Flatten the input to a 1D vector
    Dense(128, activation='relu'),  # First hidden layer with 128 neurons and ReLU activation
    Dense(10)  # Output layer with 10 neurons for the 10 classes (digits 0-9)
])


#### 3. Define the loss function, optimizer, and metric: 

- Use Sparse Categorical Crossentropy for the loss function and Adam optimizer. 

- Add Sparse Categorical Accuracy as a metric. 


In [7]:
# Step 3: Define Loss Function, Optimizer, and Metric

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)  # Loss function for multi-class classification
optimizer = tf.keras.optimizers.Adam()  # Adam optimizer for efficient training
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy()  # Metric to track accuracy during training


#### 4. Implement the custom training loop with accuracy: 

Track the accuracy during training and print it at regular intervals. 


In [8]:
# Step 4: Implement the Custom Training Loop with Accuracy

epochs = 5  # Number of epochs for training

for epoch in range(epochs):
    print(f'Start of epoch {epoch + 1}')
    
    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            # Forward pass: Compute predictions
            logits = model(x_batch_train, training=True)
            # Compute loss
            loss_value = loss_fn(y_batch_train, logits)
        
        # Compute gradients
        grads = tape.gradient(loss_value, model.trainable_weights)
        # Apply gradients to update model weights
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        
        # Update the accuracy metric
        accuracy_metric.update_state(y_batch_train, logits)

        # Log the loss and accuracy every 200 steps
        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()} Accuracy = {accuracy_metric.result().numpy()}')
    
    # Reset the metric at the end of each epoch
    accuracy_metric.reset_state()


Start of epoch 1
Epoch 1 Step 0: Loss = 2.4051737785339355 Accuracy = 0.0625
Epoch 1 Step 200: Loss = 0.3731546700000763 Accuracy = 0.8359763622283936
Epoch 1 Step 400: Loss = 0.17328760027885437 Accuracy = 0.8668952584266663
Epoch 1 Step 600: Loss = 0.20415395498275757 Accuracy = 0.882799506187439
Epoch 1 Step 800: Loss = 0.14049731194972992 Accuracy = 0.8953651785850525
Epoch 1 Step 1000: Loss = 0.41318005323410034 Accuracy = 0.9026286005973816
Epoch 1 Step 1200: Loss = 0.17869704961776733 Accuracy = 0.909268319606781
Epoch 1 Step 1400: Loss = 0.240176260471344 Accuracy = 0.9143915176391602
Epoch 1 Step 1600: Loss = 0.19566291570663452 Accuracy = 0.9179614186286926
Epoch 1 Step 1800: Loss = 0.11705504357814789 Accuracy = 0.922074556350708


2026-04-14 23:49:20.482523: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Start of epoch 2
Epoch 2 Step 0: Loss = 0.06384645402431488 Accuracy = 1.0
Epoch 2 Step 200: Loss = 0.14492137730121613 Accuracy = 0.9625310897827148
Epoch 2 Step 400: Loss = 0.08512821793556213 Accuracy = 0.9597101211547852
Epoch 2 Step 600: Loss = 0.05192141979932785 Accuracy = 0.9610545039176941
Epoch 2 Step 800: Loss = 0.07800411432981491 Accuracy = 0.9618836045265198
Epoch 2 Step 1000: Loss = 0.2306509166955948 Accuracy = 0.9624438285827637
Epoch 2 Step 1200: Loss = 0.08674062043428421 Accuracy = 0.9634939432144165
Epoch 2 Step 1400: Loss = 0.17713093757629395 Accuracy = 0.9643781185150146
Epoch 2 Step 1600: Loss = 0.12864884734153748 Accuracy = 0.9645534157752991
Epoch 2 Step 1800: Loss = 0.06729903817176819 Accuracy = 0.965158224105835


2026-04-14 23:50:11.659002: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Start of epoch 3
Epoch 3 Step 0: Loss = 0.026250438764691353 Accuracy = 1.0
Epoch 3 Step 200: Loss = 0.09489571303129196 Accuracy = 0.9755908250808716
Epoch 3 Step 400: Loss = 0.08037837594747543 Accuracy = 0.973192036151886
Epoch 3 Step 600: Loss = 0.0573185570538044 Accuracy = 0.9741576313972473
Epoch 3 Step 800: Loss = 0.035817213356494904 Accuracy = 0.9741339087486267
Epoch 3 Step 1000: Loss = 0.14253965020179749 Accuracy = 0.9744630455970764
Epoch 3 Step 1200: Loss = 0.07140762358903885 Accuracy = 0.974786639213562
Epoch 3 Step 1400: Loss = 0.11476312577724457 Accuracy = 0.9752408862113953
Epoch 3 Step 1600: Loss = 0.07986606657505035 Accuracy = 0.9753474593162537
Epoch 3 Step 1800: Loss = 0.03644354268908501 Accuracy = 0.9756211638450623


2026-04-14 23:51:01.888856: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Start of epoch 4
Epoch 4 Step 0: Loss = 0.022760562598705292 Accuracy = 1.0
Epoch 4 Step 200: Loss = 0.06312776356935501 Accuracy = 0.981809675693512
Epoch 4 Step 400: Loss = 0.05220846086740494 Accuracy = 0.9807512760162354
Epoch 4 Step 600: Loss = 0.042246270924806595 Accuracy = 0.9812291860580444
Epoch 4 Step 800: Loss = 0.025570472702383995 Accuracy = 0.9811953902244568
Epoch 4 Step 1000: Loss = 0.08789563924074173 Accuracy = 0.9814248085021973
Epoch 4 Step 1200: Loss = 0.04414805769920349 Accuracy = 0.98136967420578
Epoch 4 Step 1400: Loss = 0.07429417967796326 Accuracy = 0.981486439704895
Epoch 4 Step 1600: Loss = 0.048935070633888245 Accuracy = 0.9818082451820374
Epoch 4 Step 1800: Loss = 0.026327408850193024 Accuracy = 0.9819197654724121


2026-04-14 23:51:52.367854: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Start of epoch 5
Epoch 5 Step 0: Loss = 0.017834411934018135 Accuracy = 1.0
Epoch 5 Step 200: Loss = 0.049494657665491104 Accuracy = 0.9860074520111084
Epoch 5 Step 400: Loss = 0.038592711091041565 Accuracy = 0.9858167171478271
Epoch 5 Step 600: Loss = 0.02926897630095482 Accuracy = 0.9865328669548035
Epoch 5 Step 800: Loss = 0.02366335503757 Accuracy = 0.9864622354507446
Epoch 5 Step 1000: Loss = 0.0937538743019104 Accuracy = 0.9865759015083313
Epoch 5 Step 1200: Loss = 0.04938170686364174 Accuracy = 0.9865736961364746
Epoch 5 Step 1400: Loss = 0.04106604680418968 Accuracy = 0.9866613149642944
Epoch 5 Step 1600: Loss = 0.03693689778447151 Accuracy = 0.9866880178451538
Epoch 5 Step 1800: Loss = 0.012533467262983322 Accuracy = 0.9869690537452698


2026-04-14 23:52:44.101485: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### Exercise 3: Custom Callback for Advanced Logging: 

Implement a custom callback to log additional metrics and information during training. 

#### 1. Set Up the Environment: 

Follow the setup from Exercise 1.


In [9]:
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Flatten 

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize the pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0 

# Create a batched dataset for training
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)


#### 2. Define the Model: 

Use the same model as in Exercise 1. 


In [10]:
# Step 2: Define the Model

model = Sequential([
    Flatten(input_shape=(28, 28)),  # Flatten the input to a 1D vector
    Dense(128, activation='relu'),  # First hidden layer with 128 neurons and ReLU activation
    Dense(10)  # Output layer with 10 neurons for the 10 classes (digits 0-9)
])


#### 3. Define Loss Function, Optimizer, and Metric: 

- Use Sparse Categorical Crossentropy for the loss function and Adam optimizer. 

- Add Sparse Categorical Accuracy as a metric. 


In [11]:
# Step 3: Define Loss Function, Optimizer, and Metric

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)  # Loss function for multi-class classification
optimizer = tf.keras.optimizers.Adam()  # Adam optimizer for efficient training
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy()  # Metric to track accuracy during training


#### 4. Implement the custom training loop with custom callback: 

Create a custom callback to log additional metrics at the end of each epoch.


In [12]:
from tensorflow.keras.callbacks import Callback

# Step 4: Implement the Custom Callback 
class CustomCallback(Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        print(f'End of epoch {epoch + 1}, loss: {logs.get("loss")}, accuracy: {logs.get("accuracy")}')


In [13]:
# Step 5: Implement the Custom Training Loop with Custom Callback

epochs = 2
custom_callback = CustomCallback()  # Initialize the custom callback

for epoch in range(epochs):
    print(f'Start of epoch {epoch + 1}')
    
    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            # Forward pass: Compute predictions
            logits = model(x_batch_train, training=True)
            # Compute loss
            loss_value = loss_fn(y_batch_train, logits)
        
        # Compute gradients
        grads = tape.gradient(loss_value, model.trainable_weights)
        # Apply gradients to update model weights
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        
        # Update the accuracy metric
        accuracy_metric.update_state(y_batch_train, logits)

        # Log the loss and accuracy every 200 steps
        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()} Accuracy = {accuracy_metric.result().numpy()}')
    
    # Call the custom callback at the end of each epoch
    custom_callback.on_epoch_end(epoch, logs={'loss': loss_value.numpy(), 'accuracy': accuracy_metric.result().numpy()})
    
    # Reset the metric at the end of each epoch
    accuracy_metric.reset_state()  # Use reset_state() instead of reset_states()


Start of epoch 1
Epoch 1 Step 0: Loss = 2.291134834289551 Accuracy = 0.15625
Epoch 1 Step 200: Loss = 0.41858920454978943 Accuracy = 0.8302238583564758
Epoch 1 Step 400: Loss = 0.16914516687393188 Accuracy = 0.8635442852973938
Epoch 1 Step 600: Loss = 0.194553405046463 Accuracy = 0.8804076313972473
Epoch 1 Step 800: Loss = 0.1780019849538803 Accuracy = 0.8928292989730835
Epoch 1 Step 1000: Loss = 0.3816336989402771 Accuracy = 0.900661826133728
Epoch 1 Step 1200: Loss = 0.16218864917755127 Accuracy = 0.9070305824279785
Epoch 1 Step 1400: Loss = 0.2482326626777649 Accuracy = 0.9124955534934998
Epoch 1 Step 1600: Loss = 0.243608757853508 Accuracy = 0.9156777262687683
Epoch 1 Step 1800: Loss = 0.22758324444293976 Accuracy = 0.9198014736175537


2026-04-14 23:56:20.835904: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


End of epoch 1, loss: 0.04476435109972954, accuracy: 0.9218833446502686
Start of epoch 2
Epoch 2 Step 0: Loss = 0.13460329174995422 Accuracy = 0.96875
Epoch 2 Step 200: Loss = 0.1801241785287857 Accuracy = 0.9598880410194397
Epoch 2 Step 400: Loss = 0.09090884029865265 Accuracy = 0.9568266868591309
Epoch 2 Step 600: Loss = 0.08703059703111649 Accuracy = 0.959182620048523
Epoch 2 Step 800: Loss = 0.11018311977386475 Accuracy = 0.9596987962722778
Epoch 2 Step 1000: Loss = 0.285805881023407 Accuracy = 0.960289716720581
Epoch 2 Step 1200: Loss = 0.07310183346271515 Accuracy = 0.9611521363258362
Epoch 2 Step 1400: Loss = 0.17471753060817719 Accuracy = 0.9620583653450012
Epoch 2 Step 1600: Loss = 0.18153436481952667 Accuracy = 0.9621525406837463
Epoch 2 Step 1800: Loss = 0.13764451444149017 Accuracy = 0.9631281495094299
End of epoch 2, loss: 0.057516153901815414, accuracy: 0.9638833403587341


2026-04-14 23:57:14.973370: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### Exercise 4: Add Hidden Layers 

Next, you will add a couple of hidden layers to your model. Hidden layers help the model learn complex patterns in the data. 


In [14]:
from tensorflow.keras.layers import Input, Dense

# Define the input layer
input_layer = Input(shape=(28, 28))  # Input layer with shape (28, 28)

# Flatten the 2D images into 1D vectors before Dense layers
flatten = Flatten()(input_layer)

# Define hidden layers
hidden_layer1 = Dense(64, activation='relu')(flatten)  # First hidden layer with 64 neurons and ReLU activation
hidden_layer2 = Dense(64, activation='relu')(hidden_layer1)  # Second hidden layer with 64 neurons and ReLU activation


In the above code: 

`Flatten()` converts each 28×28 image into a 784-element 1D vector so it can be fed into Dense layers. 

`Dense(64, activation='relu')` creates a dense (fully connected) layer with 64 units and ReLU activation function. 

Each hidden layer takes the output of the previous layer as its input.


### Exercise 5: Define the output layer 

Finally, you will define the output layer. Suppose you are working on a binary classification problem, so the output layer will have one unit with a sigmoid activation function. 


In [15]:
output_layer = Dense(10, activation='softmax')(hidden_layer2)

In the above code: 

`Dense(10, activation='softmax')` creates the output layer with 10 neurons — one for each digit class (0–9). 

`softmax` activation ensures all 10 output values sum to 1, making them interpretable as class probabilities. 


### Exercise 6: Create the Model 

Now, you will create the model by specifying the input and output layers. 


In [16]:
model = Model(inputs=input_layer, outputs=output_layer)

In the above code: 

`Model(inputs=input_layer, outputs=output_layer)` creates a Keras model that connects the input layer to the output layer through the hidden layers. 


### Exercise 7: Compile the Model 

Before training the model, you need to compile it. You will specify the loss function, optimizer, and evaluation metrics. 


In [17]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # Correct for integer labels 0-9
    metrics=['accuracy']
)

In the above code: 

`optimizer='adam'` specifies the Adam optimizer, a popular choice for training neural networks. 

`loss='sparse_categorical_crossentropy'` is the correct loss for **multi-class classification** when labels are integers (0–9).

`metrics=['accuracy']` tells Keras to evaluate the model using accuracy during training. 


### Exercise 8: Train the Model 

You can now train the model on some training data. For this example, let's assume `X_train` is our training input data and `y_train` is the corresponding labels. 


In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
import numpy as np

# Load and preprocess MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0  # Normalize pixel values to [0, 1]

# Train the model
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=32
)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9175 - loss: 0.2847
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9604 - loss: 0.1308
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9713 - loss: 0.0946
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9768 - loss: 0.0745
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9808 - loss: 0.0605


In the above code: 

`X_train` and `y_train` are placeholders for your actual training data. 

`model.fit` trains the model for a specified number of epochs and batch size. 


### Exercise 9: Evaluate the Model 

After training, you can evaluate the model on test data to see how well it performs. 


In [19]:
# Example test data (in practice, use real dataset)
loss, accuracy = model.evaluate(x_test, y_test)

print(f'Test loss:     {loss:.4f}')
print(f'Test accuracy: {accuracy:.4f}')



313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9724 - loss: 0.0882
Test loss:     0.0882
Test accuracy: 0.9724


In the above code: 

`model.evaluate` computes the loss and accuracy of the model on test data. 

`X_test` and `y_test` are placeholders for your actual test data. 


## Practice Exercises 

### Exercise 1: Basic Custom Training Loop 

#### Objective: Implement a basic custom training loop to train a simple neural network on the MNIST dataset. 

#### Instructions: 

- Set up the environment and load the dataset. 

- Define the model with a Flatten layer and two Dense layers. 

- Define the loss function and optimizer. 

- Implement a custom training loop to iterate over the dataset, compute the loss, and update the model's weights. 


In [20]:
# Write your code here
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Flatten 

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data() 
x_train, x_test = x_train / 255.0, x_test / 255.0 
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32) 

# Step 2: Define the Model
model = Sequential([ 
    Flatten(input_shape=(28, 28)), 
    Dense(128, activation='relu'), 
    Dense(10) 
]) 

# Step 3: Define Loss Function and Optimizer
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) 
optimizer = tf.keras.optimizers.Adam() 

# Step 4: Implement the Custom Training Loop
for epoch in range(5): 
    for x_batch, y_batch in train_dataset: 
        with tf.GradientTape() as tape: 
            logits = model(x_batch, training=True) 
            loss = loss_fn(y_batch, logits) 
        grads = tape.gradient(loss, model.trainable_weights) 
        optimizer.apply_gradients(zip(grads, model.trainable_weights)) 
    print(f'Epoch {epoch + 1}: Loss = {loss.numpy()}')

2026-04-14 23:58:59.946549: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 1: Loss = 0.06244362145662308


2026-04-14 23:59:55.433553: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 2: Loss = 0.050483930855989456


2026-04-15 00:00:48.340469: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 3: Loss = 0.029201887547969818


2026-04-15 00:01:40.186532: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 4: Loss = 0.009173551574349403
Epoch 5: Loss = 0.00732769537717104


2026-04-15 00:02:31.249160: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


<details>
<summary>Click here for solution</summary> </br>

```python
# Import necessary libraries
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Flatten 

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data() 
x_train, x_test = x_train / 255.0, x_test / 255.0 
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32) 

# Step 2: Define the Model
model = Sequential([ 
    Flatten(input_shape=(28, 28)), 
    Dense(128, activation='relu'), 
    Dense(10) 
]) 

# Step 3: Define Loss Function and Optimizer
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) 
optimizer = tf.keras.optimizers.Adam() 

# Step 4: Implement the Custom Training Loop
for epoch in range(5): 
    for x_batch, y_batch in train_dataset: 
        with tf.GradientTape() as tape: 
            logits = model(x_batch, training=True) 
            loss = loss_fn(y_batch, logits) 
        grads = tape.gradient(loss, model.trainable_weights) 
        optimizer.apply_gradients(zip(grads, model.trainable_weights)) 
    print(f'Epoch {epoch + 1}: Loss = {loss.numpy()}')


### Exercise 2: Adding Accuracy Metric 

#### Objective: Enhance the custom training loop by adding an accuracy metric to monitor model performance. 

#### Instructions: 

1. Set up the environment and define the model, loss function, and optimizer. 

2. Add Sparse Categorical Accuracy as a metric. 

3. Implement the custom training loop with accuracy tracking.


In [21]:
# Write your code here
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Flatten 

# Step 1: Set Up the Environment
(x_train, y_train), _ = tf.keras.datasets.mnist.load_data() 
x_train = x_train / 255.0 
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32) 

# Step 2: Define the Model
model = Sequential([ 
    Flatten(input_shape=(28, 28)), 
    Dense(128, activation='relu'), 
    Dense(10) 
]) 

# Step 3: Define Loss Function, Optimizer, and Metric
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) 
optimizer = tf.keras.optimizers.Adam() 
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy() 

# Step 4: Implement the Custom Training Loop with Accuracy Tracking
epochs = 5 
for epoch in range(epochs): 
    for x_batch, y_batch in train_dataset: 
        with tf.GradientTape() as tape: 
            logits = model(x_batch, training=True) 
            loss = loss_fn(y_batch, logits) 
        grads = tape.gradient(loss, model.trainable_weights) 
        optimizer.apply_gradients(zip(grads, model.trainable_weights)) 
        accuracy_metric.update_state(y_batch, logits) 
    print(f'Epoch {epoch + 1}: Loss = {loss.numpy()} Accuracy = {accuracy_metric.result().numpy()}') 
    accuracy_metric.reset_state() 

2026-04-15 00:03:31.015158: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 1: Loss = 0.047282181680202484 Accuracy = 0.9240000247955322


2026-04-15 00:04:32.860723: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 2: Loss = 0.03810625150799751 Accuracy = 0.9637166857719421


2026-04-15 00:05:33.553588: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 3: Loss = 0.026110483333468437 Accuracy = 0.9753000140190125


2026-04-15 00:06:32.229569: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 4: Loss = 0.021234413608908653 Accuracy = 0.9818999767303467
Epoch 5: Loss = 0.008486732840538025 Accuracy = 0.9871666431427002


2026-04-15 00:07:30.555800: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


<details>
<summary>Click here for solution</summary><br>

```python
# Import necessary libraries
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Flatten 

# Step 1: Set Up the Environment
(x_train, y_train), _ = tf.keras.datasets.mnist.load_data() 
x_train = x_train / 255.0 
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32) 

# Step 2: Define the Model
model = Sequential([ 
    Flatten(input_shape=(28, 28)), 
    Dense(128, activation='relu'), 
    Dense(10) 
]) 

# Step 3: Define Loss Function, Optimizer, and Metric
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) 
optimizer = tf.keras.optimizers.Adam() 
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy() 

# Step 4: Implement the Custom Training Loop with Accuracy Tracking
epochs = 5 
for epoch in range(epochs): 
    for x_batch, y_batch in train_dataset: 
        with tf.GradientTape() as tape: 
            logits = model(x_batch, training=True) 
            loss = loss_fn(y_batch, logits) 
        grads = tape.gradient(loss, model.trainable_weights) 
        optimizer.apply_gradients(zip(grads, model.trainable_weights)) 
        accuracy_metric.update_state(y_batch, logits) 
    print(f'Epoch {epoch + 1}: Loss = {loss.numpy()} Accuracy = {accuracy_metric.result().numpy()}') 
    accuracy_metric.reset_state() 


### Exercise 3: Custom Callback for Advanced Logging 

#### Objective: Implement a custom callback to log additional metrics and information during training. 

#### Instructions: 

1. Set up the environment and define the model, loss function, optimizer, and metric. 

2. Create a custom callback to log additional metrics at the end of each epoch. 

3. Implement the custom training loop with the custom callback. 


In [22]:
# Write your code here
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Flatten 
from tensorflow.keras.callbacks import Callback 

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data() 
x_train = x_train / 255.0 
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32) 

# Step 2: Define the Model
model = Sequential([ 
    tf.keras.Input(shape=(28, 28)),  # Updated Input layer syntax
    Flatten(), 
    Dense(128, activation='relu'), 
    Dense(10) 
]) 

# Step 3: Define Loss Function, Optimizer, and Metric
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) 
optimizer = tf.keras.optimizers.Adam() 
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy() 

# Step 4: Implement the Custom Callback
class CustomCallback(Callback): 
    def on_epoch_end(self, epoch, logs=None): 
        print(f'End of epoch {epoch + 1}, loss: {logs.get("loss")}, accuracy: {logs.get("accuracy")}') 

# Step 5: Implement the Custom Training Loop with Custom Callback
custom_callback = CustomCallback() 

for epoch in range(5): 
    for x_batch, y_batch in train_dataset: 
        with tf.GradientTape() as tape: 
            logits = model(x_batch, training=True) 
            loss = loss_fn(y_batch, logits) 
        grads = tape.gradient(loss, model.trainable_weights) 
        optimizer.apply_gradients(zip(grads, model.trainable_weights)) 
        accuracy_metric.update_state(y_batch, logits) 
    custom_callback.on_epoch_end(epoch, logs={'loss': loss.numpy(), 'accuracy': accuracy_metric.result().numpy()}) 
    accuracy_metric.reset_state()

2026-04-15 00:08:38.518341: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


End of epoch 1, loss: 0.049175411462783813, accuracy: 0.9227499961853027


2026-04-15 00:09:43.990301: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


End of epoch 2, loss: 0.04982493817806244, accuracy: 0.9649333357810974


2026-04-15 00:11:56.282122: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


End of epoch 3, loss: 0.06089287996292114, accuracy: 0.9767500162124634


2026-04-15 00:13:37.401459: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


End of epoch 4, loss: 0.04970801621675491, accuracy: 0.9830666780471802
End of epoch 5, loss: 0.033990368247032166, accuracy: 0.9873999953269958


2026-04-15 00:14:53.765174: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


<details>
<summary>Click here for solution</summary> </br>

```python
# Import necessary libraries
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Flatten 
from tensorflow.keras.callbacks import Callback 

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data() 
x_train = x_train / 255.0 
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32) 

# Step 2: Define the Model
model = Sequential([ 
    tf.keras.Input(shape=(28, 28)),  # Updated Input layer syntax
    Flatten(), 
    Dense(128, activation='relu'), 
    Dense(10) 
]) 

# Step 3: Define Loss Function, Optimizer, and Metric
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) 
optimizer = tf.keras.optimizers.Adam() 
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy() 

# Step 4: Implement the Custom Callback
class CustomCallback(Callback): 
    def on_epoch_end(self, epoch, logs=None): 
        print(f'End of epoch {epoch + 1}, loss: {logs.get("loss")}, accuracy: {logs.get("accuracy")}') 

# Step 5: Implement the Custom Training Loop with Custom Callback
custom_callback = CustomCallback() 

for epoch in range(5): 
    for x_batch, y_batch in train_dataset: 
        with tf.GradientTape() as tape: 
            logits = model(x_batch, training=True) 
            loss = loss_fn(y_batch, logits) 
        grads = tape.gradient(loss, model.trainable_weights) 
        optimizer.apply_gradients(zip(grads, model.trainable_weights)) 
        accuracy_metric.update_state(y_batch, logits) 
    custom_callback.on_epoch_end(epoch, logs={'loss': loss.numpy(), 'accuracy': accuracy_metric.result().numpy()}) 
    accuracy_metric.reset_state()  # Updated method



### Exercise 5: Add Hidden layer and the Output Layer 

#### Objective: Add couple of hidden layer to help the model learn complex patterns in the data.
### Define the output layer of a neural network for a multi-class classification problem using the Keras Functional API. 

#### Instructions: 
- Define 2 hidden layers, Input layer(of shape 28,28) as the parameter for the first hidden layer)

- Using the `hidden_layer2` as the input, add a `Dense` output layer. 

- The output layer with `sigmoid function` as activation function


In [23]:
# Write your code here
from tensorflow.keras.layers import Input, Dense, Flatten

# Re-define layers for MNIST (28x28 images, 10 digit classes)
input_layer   = Input(shape=(28, 28))                          # Input: 28x28 grayscale images
flatten       = Flatten()(input_layer)                         # Flatten to 784-dim vector
hidden_layer1 = Dense(64, activation='relu')(flatten)          # First hidden layer
hidden_layer2 = Dense(64, activation='relu')(hidden_layer1)    # Second hidden layer
output_layer  = Dense(10, activation='softmax')(hidden_layer2) # Output: 10 classes (digits 0-9)

<details>
<summary>Click here for solution</summary> </br>

```python
from tensorflow.keras.layers import Input, Dense, Flatten

# Re-define layers for MNIST (28x28 images, 10 digit classes)
input_layer   = Input(shape=(28, 28))                          # Input: 28x28 grayscale images
flatten       = Flatten()(input_layer)                         # Flatten to 784-dim vector
hidden_layer1 = Dense(64, activation='relu')(flatten)          # First hidden layer
hidden_layer2 = Dense(64, activation='relu')(hidden_layer1)    # Second hidden layer
output_layer  = Dense(10, activation='softmax')(hidden_layer2) # Output: 10 classes (digits 0-9)
 ```   

</details>


### Exercise 6: Create the Model 

#### Objective: Create a Keras Functional API model by connecting the input and output layers defined in the previous exercises. 

#### Instructions: 

- Use `tf.keras.Model` to create the model. 

- Pass `input_layer` as the `inputs` argument and `output_layer` as the `outputs` argument. 


In [24]:
# Write your code here
model = Model(inputs=input_layer, outputs=output_layer)

<details>
<summary>Click here for solution</summary> </br>

```python
# Create the model by specifying input and output layers
model = Model(inputs=input_layer, outputs=output_layer)
 ```   

</details>


### Exercise 7: Compile the Model 

#### Objective: Configure the model for training by specifying the optimizer, loss function, and evaluation metric. 

#### Instructions: 

- Compile the model using the **Adam** optimizer. 

- Use **`sparse_categorical_crossentropy`** as the loss function
- Include **`accuracy`** as the evaluation metric. 


In [25]:
# Write your code here
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # Correct loss for integer labels (0-9)
    metrics=['accuracy']
)

<details>
<summary>Click here for solution</summary> </br>

```python
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # Correct loss for integer labels (0-9)
    metrics=['accuracy']
)
```

</details>


### Exercise 8: Train the Model 

#### Objective: Train the compiled model on the MNIST training dataset using `model.fit()`. 

#### Instructions: 

- Call `model.fit()` with the training data `x_train` and labels `y_train`. 

- Train for **5 epochs** with a **batch size of 32**. 


In [26]:
# Write your code here
history = model.fit(
    x_train, y_train,        # Training data and labels
    epochs=5,                # Number of full passes over the training data
    batch_size=32,           # Number of samples per gradient update
)

print('Training complete.')

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9179 - loss: 0.2806
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9619 - loss: 0.1270
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9713 - loss: 0.0936
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9776 - loss: 0.0740
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9816 - loss: 0.0598
Training complete.


<details>
<summary>Click here for solution</summary> </br>

```python
history = model.fit(
    x_train, y_train,        # Training data and labels
    epochs=5,                # Number of full passes over the training data
    batch_size=32,           # Number of samples per gradient update
)

print('Training complete.')
```

</details>


### Exercise 9: Evaluate the Model 

#### Objective: Assess the trained model's performance on unseen test data using `model.evaluate()`. 

#### Instructions: 

- Call `model.evaluate()` with the test data `x_test` and labels `y_test`. 

- Capture the returned **test loss** and **test accuracy**. 

- Print both values to summarise the model's generalisation performance. 


In [27]:
# Write your code here
test_loss, test_accuracy = model.evaluate(x_test, y_test)

# Print the evaluation results
print(f'Test loss:     {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9723 - loss: 17.2062
Test loss:     17.2062
Test accuracy: 0.9723


<details>
<summary>Click here for solution</summary> </br>

```python
# Evaluate the trained model on the held-out test dataset
test_loss, test_accuracy = model.evaluate(x_test, y_test)

# Print the evaluation results
print(f'Test loss:     {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')
```

</details>


### Conclusion: 

Congratulations on completing this lab! You have now successfully created, trained, and evaluated a simple neural network model using the Keras Functional API. This foundational knowledge will allow you to build more complex models and explore advanced functionalities in Keras. 


Copyright © IBM Corporation. All rights reserved.
